# Part 5: `test_input_sequence1.mp4` Colab GPU 자동 라벨링

포즈 티처 모델 `RTMW-x 384×288`과 `Sapiens-1B`을 같은 사람 바운딩 박스에 각각 적용합니다.

- 런타임 유형: **GPU (T4/L4/A100)**
- 비교 기준: 두 모델 모두 COCO body 17개 관절만 사용
- 시점 대응: 원본과 180° 회전 입력 중 몸통 관절 신뢰도가 높은 결과 선택
- 장당 시간: 모델별 `*_pose_ms`에 한 이미지의 전체 사람·0°/180° TTA 포즈 단계 시간을 합산
- 출력: `labels_pose_rtmw_x/`, `labels_pose_sapiens_1b/`, 모델별 COCO JSON, `frame_labels.csv`

`labels_pose/`와 `annotations_coco.json`은 기존 파이프라인 호환을 위해 RTMW-x 결과를 복제합니다.


In [ ]:
!pip -q install ultralytics opencv-python-headless numpy pandas matplotlib scipy munkres pycocotools huggingface_hub json-tricks
!pip -q install "mmengine>=0.10.0" "mmcv-lite>=2.0.1,<2.2.0"
!pip -q install --no-deps "mmpose>=1.3.0"


In [ ]:
# 방법 A: 로컬에서 test_input_sequence1.mp4 직접 업로드
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded.keys()))
print("Uploaded:", VIDEO_PATH)

# 방법 B: Google Drive 사용 시 위 주석 처리 후 아래 사용
# from google.colab import drive
# drive.mount("/content/drive")
# VIDEO_PATH = "/content/drive/MyDrive/test_input_sequence1.mp4"


In [ ]:
import json
from pathlib import Path

ROI_CFG = '{\n  "roi_version": "test_input_sequence1_perspective_trapezoid_v1",\n  "roi_source": "frame_0000 supine baseline of test_input_sequence1.mp4; board edges extended along perspective lines toward foot side",\n  "roi_reviewed": true,\n  "deployment_approved": false,\n  "bed_polygon": [\n    [0.278, 0.040],\n    [0.626, 0.040],\n    [0.882, 0.970],\n    [0.073, 0.970]\n  ],\n  "raw_board_polygon": [\n    [0.255, 0.145],\n    [0.655, 0.150],\n    [0.880, 0.962],\n    [0.075, 0.962]\n  ],\n  "perspective_extension": {\n    "far_edge_ratio": 0.1285,\n    "direction": "head_to_foot_upward",\n    "reason": "Camera views from head toward feet; feet extend past raw board top edge in image space."\n  }\n}\n'
TIMELINE_CFG = '{\n  "video": "test_input_sequence1.mp4",\n  "fps": 30.0,\n  "total_frames": 1024,\n  "roi_config": "test_input_sequence1_roi_v1.json",\n  "verification_note": "Visually verified at 15-frame (0.5s) resolution against test_input_sequence1_roi_v1.json bed_polygon.",\n  "events": [\n    {"start_sec": 0.0, "end_sec": 0.75, "start_frame": 0, "end_frame": 23, "exit_hint": false, "phase": "supine_baseline", "description": "정자세 (ROI 내부)"},\n    {"start_sec": 0.75, "end_sec": 3.25, "start_frame": 23, "end_frame": 98, "exit_hint": true, "phase": "left_side_arm_exit", "description": "화면 왼쪽으로 오른팔 뻗어 ROI 밖 이탈"},\n    {"start_sec": 3.25, "end_sec": 4.75, "start_frame": 98, "end_frame": 143, "exit_hint": false, "phase": "supine_return_1", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 4.75, "end_sec": 6.25, "start_frame": 143, "end_frame": 188, "exit_hint": true, "phase": "right_side_arm_exit", "description": "화면 오른쪽으로 왼팔 뻗어 ROI 밖 이탈"},\n    {"start_sec": 6.25, "end_sec": 7.25, "start_frame": 188, "end_frame": 218, "exit_hint": false, "phase": "supine_return_2", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 7.25, "end_sec": 8.75, "start_frame": 218, "end_frame": 263, "exit_hint": true, "phase": "all_limbs_exit", "description": "양팔·양다리 대자로 ROI 밖 이탈"},\n    {"start_sec": 8.75, "end_sec": 10.25, "start_frame": 263, "end_frame": 308, "exit_hint": false, "phase": "supine_return_3", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 10.25, "end_sec": 13.25, "start_frame": 308, "end_frame": 398, "exit_hint": true, "phase": "left_roll_shift_exit", "description": "화면 왼쪽으로 몸통·다리 쏠리며 ROI 왼쪽 경계 이탈"},\n    {"start_sec": 13.25, "end_sec": 14.75, "start_frame": 398, "end_frame": 443, "exit_hint": false, "phase": "supine_safe", "description": "정자세 (ROI 내부)"},\n    {"start_sec": 14.75, "end_sec": 17.25, "start_frame": 443, "end_frame": 518, "exit_hint": true, "phase": "right_side_leg_arm_exit", "description": "화면 오른쪽으로 다리·팔 ROI 밖 이탈"},\n    {"start_sec": 17.25, "end_sec": 18.75, "start_frame": 518, "end_frame": 563, "exit_hint": false, "phase": "sit_up_inside", "description": "일어나 앉는 자세 (상체 ROI 내부)"},\n    {"start_sec": 18.75, "end_sec": 21.75, "start_frame": 563, "end_frame": 653, "exit_hint": true, "phase": "sit_up_top_exit", "description": "앉은 상태에서 머리·상체가 상단 ROI 경계 밖 이탈"},\n    {"start_sec": 21.75, "end_sec": 22.75, "start_frame": 653, "end_frame": 683, "exit_hint": false, "phase": "lie_back_inside", "description": "다시 누우며 ROI 내부 복귀"},\n    {"start_sec": 22.75, "end_sec": 24.25, "start_frame": 683, "end_frame": 728, "exit_hint": true, "phase": "turn_left_edge_exit", "description": "돌아눕는 과정에서 팔/어깨가 왼쪽 ROI 경계 밖 이탈"},\n    {"start_sec": 24.25, "end_sec": 25.75, "start_frame": 728, "end_frame": 773, "exit_hint": false, "phase": "prone_inside", "description": "엎드린 상태 (팔꿈치 포함 ROI 내부)"},\n    {"start_sec": 25.75, "end_sec": 28.25, "start_frame": 773, "end_frame": 848, "exit_hint": true, "phase": "prone_arms_exit", "description": "엎드린 채 양팔·한팔 좌우 ROI 밖 이탈"},\n    {"start_sec": 28.25, "end_sec": 29.75, "start_frame": 848, "end_frame": 893, "exit_hint": false, "phase": "pushup_inside", "description": "양팔 모으고 상체 일으키기 직전 (ROI 내부)"},\n    {"start_sec": 29.75, "end_sec": 34.13, "start_frame": 893, "end_frame": 1024, "exit_hint": true, "phase": "stand_up_exit", "description": "상체 상단 ROI 이탈 후 기립 및 보행 이탈"}\n  ]\n}\n'

Path("roi.json").write_text(ROI_CFG, encoding="utf-8")
Path("timeline.json").write_text(TIMELINE_CFG, encoding="utf-8")
print("Saved roi.json and timeline.json")


## 비교 모델과 출력 정의

| 모델 | 모델 정의 | 원본 출력 | 이 노트북에서 사용하는 출력 | 공식 입력 크기/학습 정보 |
|---|---|---:|---:|---|
| **RTMW-x 384×288** | MMPose의 대형 top-down whole-body pose 모델 | 133개 whole-body 관절 | 앞 17개 COCO body 관절 | 384×288, Cocktail14 학습; 공식 표의 body AP 0.763 / whole-body AP 0.702 |
| **Sapiens-1B** | Meta Sapiens의 10억 파라미터 human pose 모델 | 17개 COCO body 관절 | 17개 COCO body 관절 | 1024×768, 3억 장 human image 사전학습; 공식 COCO AP 82.1 |

두 AP 수치는 학습 데이터와 평가 프로토콜이 달라 직접 우열 비교값으로 사용하지 않습니다. 실제 선택은 동일 영상에서의 검출 누락, ROI 이탈 판정 오류, 속도로 비교해야 합니다.

Sapiens-1B 체크포인트는 약 **4.68GB**이며 Hugging Face의 모델 사용조건 승인과 로그인이 필요합니다. 라이선스는 **CC BY-NC 4.0**이므로 상업적 사용 전 별도 검토가 필요합니다.


In [ ]:
import csv, json
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import torch
from ultralytics import YOLO
import sys
import types
import pycocotools.coco as _pycoco
_xt = types.ModuleType("xtcocotools")
_xt_coco = types.ModuleType("xtcocotools.coco")
_xt_coco.COCO = _pycoco.COCO
_xt.coco = _xt_coco
sys.modules.setdefault("xtcocotools", _xt)
sys.modules.setdefault("xtcocotools.coco", _xt_coco)
from mmpose.apis import MMPoseInferencer

STEP = 1
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("test_input_sequence1_labeled")
print("Device:", DEVICE)

roi_pts = np.asarray(json.loads(Path("roi.json").read_text())["bed_polygon"], dtype=np.float32)
timeline_events = json.loads(Path("timeline.json").read_text())["events"]

COCO_COMMON = (0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16)
TORSO = (5, 6, 11, 12)

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def lookup_timeline(frame_idx):
    for ev in timeline_events:
        if ev["start_frame"] <= frame_idx < ev["end_frame"]:
            hint = "EXIT" if ev["exit_hint"] is True else ("SAFE" if ev["exit_hint"] is False else "END")
            return ev["phase"], hint
    return "unknown", "UNKNOWN"

def roi_contains(pt, shape, tol=3.0):
    h, w = shape[:2]
    poly = roi_pts * np.array([w - 1, h - 1], np.float32)
    return cv2.pointPolygonTest(poly, (float(pt[0]), float(pt[1])), True) >= -tol

def pose_exit_hint(pts, shape):
    usable = np.isfinite(pts).all(axis=1) & (pts[:, 2] >= 0.5)
    if sum(usable[i] for i in TORSO) < 2:
        return "UNAVAIL"
    selected = pts[[i for i in COCO_COMMON if usable[i]], :2]
    if any(not roi_contains(point, shape) for point in selected):
        return "EXIT"
    if not all(usable[i] for i in COCO_COMMON):
        return "UNAVAIL"
    return "SAFE"

def to_yolo_pose_line(xywhn, keypoints, scores, width, height):
    bx, by, bw, bh = (float(value) for value in xywhn)
    tokens = []
    for index in range(17):
        x_px, y_px = (float(value) for value in keypoints[index, :2])
        confidence = float(scores[index])
        x_norm = np.clip(x_px / max(width - 1, 1), 0.0, 1.0)
        y_norm = np.clip(y_px / max(height - 1, 1), 0.0, 1.0)
        visibility = 2 if confidence >= 0.5 else (1 if confidence > 0.1 else 0)
        tokens.append(f"{x_norm:.6f} {y_norm:.6f} {visibility}")
    return f"0 {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f} " + " ".join(tokens)

def to_coco_keypoints(keypoints, scores):
    values = []
    for index in range(17):
        confidence = float(scores[index])
        visibility = 2 if confidence >= 0.5 else (1 if confidence > 0.1 else 0)
        values.extend([float(keypoints[index, 0]), float(keypoints[index, 1]), visibility])
    return values

def infer_rtmw_once(crop):
    sync_device()
    started = perf_counter()
    result = next(rtmw_teacher(crop, return_vis=False))
    sync_device()
    elapsed_ms = (perf_counter() - started) * 1000.0

    predictions = result["predictions"][0]
    if not predictions:
        return None, elapsed_ms
    prediction = max(
        predictions,
        key=lambda item: float(np.asarray(item["keypoint_scores"], dtype=np.float32).mean()),
    )
    keypoints = np.asarray(prediction["keypoints"], dtype=np.float32)
    scores = np.asarray(prediction["keypoint_scores"], dtype=np.float32)
    if keypoints.ndim == 3:
        keypoints = keypoints[0]
    if scores.ndim == 2:
        scores = scores[0]
    return (keypoints[:17, :2], scores[:17]), elapsed_ms

def infer_rtmw_tta(frame, bbox, pad_ratio=0.10):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = (float(value) for value in bbox[:4])
    pad_x = (x2 - x1) * pad_ratio
    pad_y = (y2 - y1) * pad_ratio
    left = max(0, int(np.floor(x1 - pad_x)))
    top = max(0, int(np.floor(y1 - pad_y)))
    right = min(w, int(np.ceil(x2 + pad_x)))
    bottom = min(h, int(np.ceil(y2 + pad_y)))
    crop = frame[top:bottom, left:right]
    if crop.size == 0:
        return None, 0.0, 0

    variants = []
    total_ms = 0.0
    passes = 0
    for rotated, pose_input in (
        (False, crop),
        (True, cv2.rotate(crop, cv2.ROTATE_180)),
    ):
        pose, elapsed_ms = infer_rtmw_once(pose_input)
        total_ms += elapsed_ms
        passes += 1
        if pose is None:
            continue
        keypoints, scores = pose
        if rotated:
            crop_h, crop_w = crop.shape[:2]
            keypoints[:, 0] = crop_w - 1 - keypoints[:, 0]
            keypoints[:, 1] = crop_h - 1 - keypoints[:, 1]
        keypoints[:, 0] += left
        keypoints[:, 1] += top
        torso_quality = float(scores[list(TORSO)].mean())
        variants.append((torso_quality, keypoints, scores))

    if not variants:
        return None, total_ms, passes
    _, keypoints, scores = max(variants, key=lambda item: item[0])
    return (keypoints, scores), total_ms, passes

for subdir in (
    "images", "labels_bbox", "labels_seg", "labels_pose",
    "labels_pose_rtmw_x", "labels_pose_sapiens_1b",
):
    (OUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

det_yolo = YOLO("yolo11x.pt")
seg_yolo = YOLO("yolo11x-seg.pt")
RTMW_TEACHER = "rtmw-x_8xb320-270e_cocktail14-384x288"
rtmw_teacher = MMPoseInferencer(
    pose2d=RTMW_TEACHER,
    det_model="whole_image",
    device=DEVICE,
)
print("RTMW-x teacher:", RTMW_TEACHER, "| output: first 17 COCO body keypoints | TTA: 0/180 degrees")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
coco_rtmw = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "person", "supercategory": "person"}],
}
csv_rows, ann_id, frame_idx, saved = [], 1, 0, 0

while True:
    ok, frame = cap.read()
    if not ok or frame is None:
        break
    if frame_idx % STEP != 0:
        frame_idx += 1
        continue

    h, w = frame.shape[:2]
    stem = f"frame_{frame_idx:04d}"
    image_name = f"{stem}.jpg"
    cv2.imwrite(str(OUT_DIR / "images" / image_name), frame)

    det_result = det_yolo.predict(
        source=frame, classes=[0], conf=0.25, imgsz=640, device=DEVICE, verbose=False
    )[0]
    bbox_lines = []
    det_exit = "UNAVAIL"
    if det_result.boxes is not None and len(det_result.boxes) > 0:
        for xywhn in det_result.boxes.xywhn.cpu().numpy():
            bbox_lines.append(
                f"0 {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}"
            )
        first_bbox = det_result.boxes.xyxy.cpu().numpy()[0]
        center = ((first_bbox[0] + first_bbox[2]) / 2, (first_bbox[1] + first_bbox[3]) / 2)
        det_exit = "SAFE" if roi_contains(center, frame.shape, 0.0) else "EXIT"
    (OUT_DIR / "labels_bbox" / f"{stem}.txt").write_text("\n".join(bbox_lines), encoding="utf-8")

    seg_result = seg_yolo.predict(
        source=frame, classes=[0], conf=0.25, imgsz=640, device=DEVICE, verbose=False
    )[0]
    seg_lines = []
    if seg_result.masks is not None and seg_result.masks.xyn is not None:
        for polygon in seg_result.masks.xyn:
            if len(polygon) >= 3:
                seg_lines.append(
                    "0 " + " ".join(f"{point[0]:.6f} {point[1]:.6f}" for point in polygon)
                )
    (OUT_DIR / "labels_seg" / f"{stem}.txt").write_text("\n".join(seg_lines), encoding="utf-8")

    pose_lines = []
    pose_exit = "UNAVAIL"
    pose_ms = 0.0
    pose_passes = 0
    pose_people = 0
    coco_rtmw["images"].append(
        {"id": frame_idx + 1, "file_name": image_name, "width": w, "height": h}
    )

    if det_result.boxes is not None and len(det_result.boxes) > 0:
        xywhn_all = det_result.boxes.xywhn.cpu().numpy()
        xyxy_all = det_result.boxes.xyxy.cpu().numpy()
        first_pose = None
        for index in range(len(xywhn_all)):
            pose, elapsed_ms, passes = infer_rtmw_tta(frame, xyxy_all[index])
            pose_ms += elapsed_ms
            pose_passes += passes
            if pose is None:
                continue
            pose_people += 1
            keypoints, scores = pose
            if first_pose is None:
                first_pose = np.concatenate([keypoints, scores[:, None]], axis=1)
            pose_lines.append(to_yolo_pose_line(xywhn_all[index], keypoints, scores, w, h))

            coco_keypoints = to_coco_keypoints(keypoints, scores)
            x1, y1, x2, y2 = (float(value) for value in xyxy_all[index])
            coco_rtmw["annotations"].append({
                "id": ann_id,
                "image_id": frame_idx + 1,
                "category_id": 1,
                "bbox": [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)],
                "area": max(0.0, (x2 - x1) * (y2 - y1)),
                "iscrowd": 0,
                "num_keypoints": sum(1 for value in coco_keypoints[2::3] if value > 0),
                "keypoints": coco_keypoints,
            })
            ann_id += 1
        if first_pose is not None:
            pose_exit = pose_exit_hint(first_pose, frame.shape)

    pose_text = "\n".join(pose_lines)
    (OUT_DIR / "labels_pose_rtmw_x" / f"{stem}.txt").write_text(pose_text, encoding="utf-8")
    (OUT_DIR / "labels_pose" / f"{stem}.txt").write_text(pose_text, encoding="utf-8")

    phase, timeline_hint = lookup_timeline(frame_idx)
    csv_rows.append({
        "frame_idx": frame_idx,
        "image": image_name,
        "timestamp_sec": f"{frame_idx / fps:.3f}",
        "timeline_phase": phase,
        "timeline_exit_hint": timeline_hint,
        "auto_exit_det_center": det_exit,
        "auto_exit_pose_rtmw_x": pose_exit,
        "rtmw_x_pose_ms": f"{pose_ms:.3f}",
        "rtmw_x_people": pose_people,
        "rtmw_x_tta_passes": pose_passes,
    })
    frame_idx += 1
    saved += 1

cap.release()
rtmw_json = json.dumps(coco_rtmw, indent=2)
(OUT_DIR / "annotations_coco_rtmw_x.json").write_text(rtmw_json, encoding="utf-8")
(OUT_DIR / "annotations_coco.json").write_text(rtmw_json, encoding="utf-8")
with (OUT_DIR / "frame_labels.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(csv_rows[0].keys()))
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"RTMW-x done: {saved} images | labels_pose_rtmw_x/ | annotations_coco_rtmw_x.json")
print("Per-image timing column: frame_labels.csv -> rtmw_x_pose_ms")


## Sapiens-1B 체크포인트 인증 및 다운로드

1. 먼저 [Sapiens pose checkpoint 페이지](https://huggingface.co/noahcao/sapiens-pose-coco)에서 사용조건에 동의합니다.
2. 아래 셀의 로그인 창에서 접근 권한이 있는 Hugging Face 토큰을 입력합니다.
3. RTMW-x 객체를 GPU에서 해제한 뒤 약 4.68GB의 Sapiens-1B TorchScript 체크포인트를 로드합니다.


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, notebook_login

!test -d /content/sapiens/.git || git clone --depth 1 https://github.com/facebookresearch/sapiens.git /content/sapiens

notebook_login()
SAPIENS_REPO = "noahcao/sapiens-pose-coco"
SAPIENS_FILE = "sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_best_coco_AP_821_torchscript.pt2"
SAPIENS_CKPT = hf_hub_download(repo_id=SAPIENS_REPO, filename=SAPIENS_FILE)
print("Sapiens-1B checkpoint:", SAPIENS_CKPT)


In [ ]:
import gc
import sys
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "/content/sapiens/lite/demo")
from pose_utils import top_down_affine_transform, udp_decode

if "rtmw_teacher" in globals():
    del rtmw_teacher
if "det_yolo" in globals():
    del det_yolo
if "seg_yolo" in globals():
    del seg_yolo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SAPIENS_INPUT_SHAPE = (1024, 768)
SAPIENS_HEATMAP_SHAPE = (256, 192)
SAPIENS_MEAN = torch.tensor([123.5, 116.5, 103.5], dtype=torch.float32).view(3, 1, 1)
SAPIENS_STD = torch.tensor([58.5, 57.0, 57.5], dtype=torch.float32).view(3, 1, 1)

sapiens_model = torch.jit.load(SAPIENS_CKPT, map_location=DEVICE).eval()
print("Sapiens-1B loaded | output: 17 COCO body keypoints | TTA: 0/180 degrees")

def infer_sapiens_once(frame, bbox):
    sync_device()
    started = perf_counter()

    warped, center, scale = top_down_affine_transform(
        frame.copy(), np.asarray(bbox, dtype=np.float32)
    )
    warped = cv2.resize(
        warped, (SAPIENS_INPUT_SHAPE[1], SAPIENS_INPUT_SHAPE[0]), interpolation=cv2.INTER_LINEAR
    )
    tensor = torch.from_numpy(warped.transpose(2, 0, 1)).float()[[2, 1, 0]]
    tensor = ((tensor - SAPIENS_MEAN) / SAPIENS_STD).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        heatmaps = sapiens_model(tensor)
    if isinstance(heatmaps, (tuple, list)):
        heatmaps = heatmaps[0]
    heatmap = heatmaps[0].detach().float().cpu().numpy()
    keypoints, scores = udp_decode(
        heatmap, SAPIENS_INPUT_SHAPE, SAPIENS_HEATMAP_SHAPE
    )

    center = np.asarray(center, dtype=np.float32).reshape(-1, 2)[0]
    scale = np.asarray(scale, dtype=np.float32).reshape(-1, 2)[0]
    keypoints = np.asarray(keypoints, dtype=np.float32).reshape(-1, 2)
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    keypoints = (
        keypoints / np.asarray(SAPIENS_INPUT_SHAPE, dtype=np.float32)
    ) * scale + center - 0.5 * scale

    sync_device()
    elapsed_ms = (perf_counter() - started) * 1000.0
    return (keypoints[:17], scores[:17]), elapsed_ms

def infer_sapiens_tta(frame, bbox):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = (float(value) for value in bbox[:4])
    rotated_bbox = np.array(
        [w - 1 - x2, h - 1 - y2, w - 1 - x1, h - 1 - y1], dtype=np.float32
    )
    variants = []
    total_ms = 0.0
    for rotated, pose_frame, pose_bbox in (
        (False, frame, np.asarray(bbox, dtype=np.float32)),
        (True, cv2.rotate(frame, cv2.ROTATE_180), rotated_bbox),
    ):
        pose, elapsed_ms = infer_sapiens_once(pose_frame, pose_bbox)
        total_ms += elapsed_ms
        keypoints, scores = pose
        if rotated:
            keypoints[:, 0] = w - 1 - keypoints[:, 0]
            keypoints[:, 1] = h - 1 - keypoints[:, 1]
        torso_quality = float(scores[list(TORSO)].mean())
        variants.append((torso_quality, keypoints, scores))

    _, keypoints, scores = max(variants, key=lambda item: item[0])
    return (keypoints, scores), total_ms, 2

def yolo_bbox_to_xyxy(line, width, height):
    _, cx, cy, bw, bh = map(float, line.split())
    return np.array([
        (cx - bw / 2) * width,
        (cy - bh / 2) * height,
        (cx + bw / 2) * width,
        (cy + bh / 2) * height,
    ], dtype=np.float32), np.array([cx, cy, bw, bh], dtype=np.float32)

frame_df = pd.read_csv(OUT_DIR / "frame_labels.csv")
sapiens_ms_values = []
sapiens_people_values = []
sapiens_pass_values = []
sapiens_exit_values = []
coco_sapiens = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "person", "supercategory": "person"}],
}
sapiens_ann_id = 1

for row in frame_df.itertuples(index=False):
    frame_idx = int(row.frame_idx)
    image_path = OUT_DIR / "images" / row.image
    frame = cv2.imread(str(image_path))
    h, w = frame.shape[:2]
    stem = image_path.stem
    bbox_path = OUT_DIR / "labels_bbox" / f"{stem}.txt"
    bbox_lines = [line for line in bbox_path.read_text().splitlines() if line.strip()]

    pose_lines = []
    first_pose = None
    frame_pose_ms = 0.0
    frame_people = 0
    frame_passes = 0
    coco_sapiens["images"].append(
        {"id": frame_idx + 1, "file_name": row.image, "width": w, "height": h}
    )

    for bbox_line in bbox_lines:
        bbox, xywhn = yolo_bbox_to_xyxy(bbox_line, w, h)
        pose, elapsed_ms, passes = infer_sapiens_tta(frame, bbox)
        frame_pose_ms += elapsed_ms
        frame_passes += passes
        frame_people += 1
        keypoints, scores = pose
        if first_pose is None:
            first_pose = np.concatenate([keypoints, scores[:, None]], axis=1)
        pose_lines.append(to_yolo_pose_line(xywhn, keypoints, scores, w, h))

        coco_keypoints = to_coco_keypoints(keypoints, scores)
        x1, y1, x2, y2 = (float(value) for value in bbox)
        coco_sapiens["annotations"].append({
            "id": sapiens_ann_id,
            "image_id": frame_idx + 1,
            "category_id": 1,
            "bbox": [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)],
            "area": max(0.0, (x2 - x1) * (y2 - y1)),
            "iscrowd": 0,
            "num_keypoints": sum(1 for value in coco_keypoints[2::3] if value > 0),
            "keypoints": coco_keypoints,
        })
        sapiens_ann_id += 1

    (OUT_DIR / "labels_pose_sapiens_1b" / f"{stem}.txt").write_text(
        "\n".join(pose_lines), encoding="utf-8"
    )
    sapiens_ms_values.append(round(frame_pose_ms, 3))
    sapiens_people_values.append(frame_people)
    sapiens_pass_values.append(frame_passes)
    sapiens_exit_values.append(
        pose_exit_hint(first_pose, frame.shape) if first_pose is not None else "UNAVAIL"
    )

frame_df["auto_exit_pose_sapiens_1b"] = sapiens_exit_values
frame_df["sapiens_1b_pose_ms"] = sapiens_ms_values
frame_df["sapiens_1b_people"] = sapiens_people_values
frame_df["sapiens_1b_tta_passes"] = sapiens_pass_values
frame_df.to_csv(OUT_DIR / "frame_labels.csv", index=False)
(OUT_DIR / "annotations_coco_sapiens_1b.json").write_text(
    json.dumps(coco_sapiens, indent=2), encoding="utf-8"
)

print(f"Sapiens-1B done: {len(frame_df)} images | labels_pose_sapiens_1b/ | annotations_coco_sapiens_1b.json")
print("Per-image timing column: frame_labels.csv -> sapiens_1b_pose_ms")


In [ ]:
import numpy as np
import pandas as pd

csv_path = OUT_DIR / "frame_labels.csv"
df = pd.read_csv(csv_path)

model_table = pd.DataFrame([
    {
        "model": "RTMW-x 384x288",
        "type": "top-down whole-body pose",
        "native_output": "133 keypoints",
        "comparison_output": "COCO body 17",
        "training_or_pretrain": "Cocktail14",
        "official_metric": "body AP 0.763 / whole AP 0.702",
    },
    {
        "model": "Sapiens-1B",
        "type": "1B-parameter human pose model",
        "native_output": "COCO body 17",
        "comparison_output": "COCO body 17",
        "training_or_pretrain": "300M in-the-wild human images pretraining",
        "official_metric": "COCO AP 82.1",
    },
])
display(model_table)

timing_rows = []
for model_name, column in (
    ("RTMW-x 384x288", "rtmw_x_pose_ms"),
    ("Sapiens-1B", "sapiens_1b_pose_ms"),
):
    if column not in df.columns:
        continue
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    nonempty = values[values > 0]
    timing_rows.append({
        "model": model_name,
        "images": int(len(values)),
        "images_with_person": int(len(nonempty)),
        "mean_ms/image": round(float(nonempty.mean()), 3) if len(nonempty) else np.nan,
        "median_ms/image": round(float(nonempty.median()), 3) if len(nonempty) else np.nan,
        "p95_ms/image": round(float(nonempty.quantile(0.95)), 3) if len(nonempty) else np.nan,
        "effective_fps": round(1000.0 / float(nonempty.mean()), 3) if len(nonempty) else np.nan,
    })

print(f"Device: {DEVICE} | total images: {len(df)}")
print("Timing includes pose preprocessing + model forward + decoding for every detected person and both 0/180-degree TTA passes.")
print("Timing excludes shared YOLO detection, segmentation, image I/O, and first-time model download.")
display(pd.DataFrame(timing_rows))
display(df[[
    "image", "rtmw_x_pose_ms", "sapiens_1b_pose_ms",
    "auto_exit_pose_rtmw_x", "auto_exit_pose_sapiens_1b",
]].head(20))


## 모델별 라벨 시각화 검토

같은 프레임을 좌우로 표시합니다. 왼쪽은 RTMW-x, 오른쪽은 Sapiens-1B입니다. 빨간 영역은 침상 ROI이며 각 제목에 모델별 장당 포즈 시간과 ROI 판정을 표시합니다.


In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np

def draw_common(image, stem):
    h, w = image.shape[:2]
    result = image.copy()
    roi_polygon = (roi_pts * np.array([w, h])).astype(np.int32)
    overlay = result.copy()
    cv2.fillPoly(overlay, [roi_polygon], (255, 0, 0))
    cv2.addWeighted(overlay, 0.18, result, 0.82, 0, result)
    cv2.polylines(result, [roi_polygon], True, (255, 0, 0), 2)

    bbox_path = OUT_DIR / "labels_bbox" / f"{stem}.txt"
    for line in bbox_path.read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(result, (x1, y1), (x2, y2), (255, 255, 0), 2)
    return result

def draw_pose(image, pose_path, color):
    h, w = image.shape[:2]
    if not pose_path.exists():
        return image
    for line in pose_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 56:
            continue
        keypoint_tokens = parts[5:]
        for index in range(17):
            x = int(float(keypoint_tokens[index * 3]) * w)
            y = int(float(keypoint_tokens[index * 3 + 1]) * h)
            visibility = int(keypoint_tokens[index * 3 + 2])
            if visibility > 0:
                cv2.circle(image, (x, y), 4, color, -1)
    return image

def visualize_model_comparison(frame_num=None):
    image_paths = sorted((OUT_DIR / "images").glob("*.jpg"))
    if not image_paths:
        print("No extracted images.")
        return
    image_path = random.choice(image_paths) if frame_num is None else OUT_DIR / "images" / f"frame_{frame_num:04d}.jpg"
    if not image_path.exists():
        image_path = image_paths[0]

    stem = image_path.stem
    bgr = cv2.imread(str(image_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    left = draw_pose(draw_common(rgb, stem), OUT_DIR / "labels_pose_rtmw_x" / f"{stem}.txt", (0, 255, 255))
    right = draw_pose(draw_common(rgb, stem), OUT_DIR / "labels_pose_sapiens_1b" / f"{stem}.txt", (255, 0, 255))

    row = df[df["image"] == image_path.name].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(left)
    axes[0].set_title(f"RTMW-x | {row['rtmw_x_pose_ms']:.1f} ms/image | {row['auto_exit_pose_rtmw_x']}")
    axes[1].imshow(right)
    axes[1].set_title(f"Sapiens-1B | {row['sapiens_1b_pose_ms']:.1f} ms/image | {row['auto_exit_pose_sapiens_1b']}")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(image_path.name)
    plt.show()

visualize_model_comparison(frame_num=150)


## 두 모델이 오버레이된 검토용 동영상

RTMW-x 관절은 노란색, Sapiens-1B 관절은 자홍색으로 표시합니다. 상단에는 프레임별 모델 추론 시간을 표시합니다.


In [ ]:
from base64 import b64encode
from pathlib import Path

import cv2
import numpy as np
from IPython.display import HTML, display

def create_annotated_video(output_filename="annotated_result.mp4", max_frames=350):
    image_paths = sorted((OUT_DIR / "images").glob("*.jpg"))
    if not image_paths:
        print("No extracted images.")
        return None

    first_image = cv2.imread(str(image_paths[0]))
    h, w = first_image.shape[:2]
    writer = cv2.VideoWriter(
        output_filename, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    for image_path in image_paths[:max_frames]:
        stem = image_path.stem
        image = cv2.imread(str(image_path))
        roi_polygon = (roi_pts * np.array([w, h])).astype(np.int32)
        overlay = image.copy()
        cv2.fillPoly(overlay, [roi_polygon], (0, 0, 255))
        cv2.addWeighted(overlay, 0.15, image, 0.85, 0, image)
        cv2.polylines(image, [roi_polygon], True, (0, 0, 255), 2)

        for pose_dir, color in (
            ("labels_pose_rtmw_x", (0, 255, 255)),
            ("labels_pose_sapiens_1b", (255, 0, 255)),
        ):
            pose_path = OUT_DIR / pose_dir / f"{stem}.txt"
            if not pose_path.exists():
                continue
            for line in pose_path.read_text().splitlines():
                parts = line.split()
                if len(parts) < 56:
                    continue
                tokens = parts[5:]
                for index in range(17):
                    x = int(float(tokens[index * 3]) * w)
                    y = int(float(tokens[index * 3 + 1]) * h)
                    if int(tokens[index * 3 + 2]) > 0:
                        cv2.circle(image, (x, y), 4, color, -1)

        row = df[df["image"] == image_path.name].iloc[0]
        text = (
            f"{image_path.name} | RTMW-x {row['rtmw_x_pose_ms']:.1f}ms | "
            f"Sapiens-1B {row['sapiens_1b_pose_ms']:.1f}ms"
        )
        cv2.putText(image, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(image)

    writer.release()
    h264_filename = "play_" + output_filename
    !ffmpeg -y -i {output_filename} -vcodec libx264 -f mp4 {h264_filename} > /dev/null 2>&1
    return h264_filename

playable_video = create_annotated_video()
if playable_video:
    video_data = Path(playable_video).read_bytes()
    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()
    display(HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>'))


In [ ]:
!zip -qr test_input_sequence1_labeled.zip test_input_sequence1_labeled/
from google.colab import files
files.download("test_input_sequence1_labeled.zip")
